# ETL (Extracción y Limpieza)

Objetivo: generar datos sintéticos con la app o por API y realizar limpieza mínima.

In [ ]:
from pathlib import Path
import pandas as pd
import json
from src.utils.reporting import write_report, log_line

# Configuración de sesión (ruta a la última sesión generada)
base_outputs = Path('outputs')
# Ajustar a la carpeta de sesión deseada
session_dirs = sorted([p for p in base_outputs.glob('session_*') if p.is_dir()])
assert session_dirs, 'No hay sesiones en outputs/. Generar datos primero con la app.'
session = session_dirs[-1]
print('Usando sesión:', session)

# Cargar metadatos
meta = json.loads((session/'session_metadata.json').read_text(encoding='utf-8'))
meta

In [ ]:
# Encontrar archivos de datos
files = list(session.glob('*.csv')) + list(session.glob('*.parquet')) + list(session.glob('*.json'))
len(files), files[:5]

In [ ]:
# Leer un archivo de ejemplo a DataFrame
sample = None
for f in files:
    if f.suffix=='.csv':
        sample = pd.read_csv(f)
        break
    elif f.suffix=='.parquet':
        sample = pd.read_parquet(f)
        break
    elif f.suffix=='.json':
        sample = pd.read_json(f)
        break

assert sample is not None, 'No se pudo cargar un archivo de ejemplo.'
sample.head()

In [ ]:
# Limpieza mínima: drops de duplicados y trimming de strings
df = sample.copy()
df = df.drop_duplicates()
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].astype(str).str.strip()

# Guardar limpio
outdir = Path('data/processed')
outdir.mkdir(parents=True, exist_ok=True)
clean_path = outdir/'clean_data.csv'
df.to_csv(clean_path, index=False)
write_report('reports/etl_report.md', 'Reporte ETL', {'rows': len(df)}, 'Se guardó clean_data.csv.')
print('Guardado:', clean_path)